# 👁️ VisionAid: Real-Time Perception & Assistive Navigation Pipeline
**Module:** Object Detection, Spatial Proximity Estimation & Acoustic Alerting  
**Target:** Assistive tech prototype for visually impaired individuals  
**Environment:** Google Colab / PyTorch / Ultralytics  

---

## 📌 Pipeline Flowchart
`Camera Input (Laptop/Mobile)`  
⬇️  
`YOLO26n Real-time Inference (Selective Ontology)`  
⬇️  
`Spatial Bounding-Box Geometry & Proximity Estimation`  
⬇️  
`Heuristic Warning & Safety Logic`  
⬇️  
`Natural Speech Synthesis (Audio Warning Output)`

## 🧠 Dataset & Perception Ontology Strategy

### 1. Current Phase (Foundation Baseline)
For the current validation stage, the system utilizes the **COCO Pretrained Weights (`yolo26n.pt`)**, which support 80 baseline classes. To avoid cognitive overload and eliminate irrelevant warnings, we applied **Strict Semantic Filtering** focusing exclusively on high-priority navigation and collision hazards:
- **Pedestrian Safety:** `Person`
- **Vehicular Obstacles:** `Car`, `Motorcycle`
- **Indoor Path Hazards:** `Chair`, `Couch`, `Bed`, `Dining Table`
- **Safety Critical & Sanitation:** `Oven` (heat hazard), `Sink`, `Toilet`, `Refrigerator`

### 2. Limitations of COCO for Assistive Navigation
Standard benchmarks (such as COCO) lack essential contextual cues required by visually impaired individuals, specifically:
- Critical negative/vertical obstacles (e.g., `Stairs`, `Curbs`).
- Navigational boundaries (e.g., `Doors`, `Glass Walls`, `Poles`).

### 3. Next Version Roadmap (Custom Dataset v0.2)
In the upcoming phase, we will transition from raw COCO weights to a fine-tuned domain-specific model (`best.pt`):
- **Curated Dataset Integration:** Merging open-source indoor navigation datasets with custom-annotated frames.
- **Novel Class Expansion:** Training the network on targeted assistive classes (`Stairs`, `Door`, `Pole`, `Crosswalk`).
- **Transfer Learning:** Fine-tuning `yolo26n` backbone weights on our custom `data.yaml` to ensure low latency and task-oriented accuracy.

In [ ]:
# ============================================================
# Cell 0: Environment Dependencies Installation
# ============================================================
import sys

# Install all core dependencies required for VisionAid pipeline
!{sys.executable} -m pip install -q --upgrade pip
!{sys.executable} -m pip install -q opencv-python ultralytics torch torchvision pillow pyttsx3 numpy

print("✅ All dependencies installed successfully.")

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [ ]:
# ============================================================
# Cell 1: Environment Setup & Package Verification
# ============================================================
import os
import sys
import time
import base64
import numpy as np
import cv2
import torch
from PIL import Image as PILImage
from IPython.display import display, Javascript
from google.colab.output import eval_js

# Ensure ultralytics is installed
try:
    from ultralytics import YOLO
except ImportError:
    !pip install -q ultralytics
    from ultralytics import YOLO

# Hardware acceleration verification
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("=" * 60)
print(f"VisionAid Environment Status:")
print(f"Python Version    : {sys.version.split()[0]}")
print(f"PyTorch Version   : {torch.__version__}")
print(f"Compute Device    : {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
print("=" * 60)

ModuleNotFoundError: No module named 'cv2'

In [ ]:
# ============================================================
# Cell 2: Model Loading & Targeted Ontology Filtering
# ============================================================
MODEL_PATH = "/content/visionaid/experiments/EXP-002_YOLO26n_v02_COCO/weights/best.pt"
if not os.path.exists(MODEL_PATH):
    print("⚠️ Custom weights (best.pt) not found. Falling back to base yolo26n.pt...")
    MODEL_PATH = "yolo26n.pt"

model = YOLO(MODEL_PATH)

# Selective Class Ontology Filtering (IDs based on COCO mapping)
# 0: person | 2: car | 3: motorcycle | 56: chair | 57: couch | 59: bed | 60: dining table | 61: toilet | 62: tv | 69: oven | 71: sink
ACTIVE_ONTOLOGY_IDS = [0, 2, 3, 56, 57, 59, 60, 61, 62, 69, 71]

print(f"Loaded Model       : {MODEL_PATH}")
print(f"Active Filter Classes: {[model.names[i] for i in ACTIVE_ONTOLOGY_IDS]}")

⚠️ Custom weights (best.pt) not found. Falling back to base yolo26n.pt...
Loaded Model       : yolo26n.pt
Active Filter Classes: ['person', 'car', 'motorcycle', 'chair', 'couch', 'bed', 'dining table', 'toilet', 'tv', 'oven', 'sink']


In [ ]:
# ============================================================
# Cell 3: Spatial Proximity & Heuristic Warning Logic
# ============================================================
def compute_spatial_hazard(box, frame_shape):
    """
    Computes normalized bounding-box geometry and proximity hazard level.
    Note: Represents relative image-space occupancy, not physical metric depth.
    """
    img_h, img_w, _ = frame_shape
    total_area = img_h * img_w

    x1, y1, x2, y2 = box.xyxy[0].tolist()
    width = max(0.0, x2 - x1)
    height = max(0.0, y2 - y1)
    box_area = width * height
    norm_area = box_area / total_area
    bottom_y = y2 / img_h  # Vertical ground-plane indicator

    # Heuristic criteria for hazard assessment
    is_critical = (norm_area > 0.22) or (bottom_y > 0.85 and norm_area > 0.12)
    is_moderate = (norm_area > 0.08)

    return {
        "norm_area": norm_area,
        "is_critical": is_critical,
        "is_moderate": is_moderate,
        "bbox": (int(x1), int(y1), int(x2), int(y2))
    }

In [ ]:
# ============================================================
# Cell 4: Single-Frame Capture & Verification Gate
# ============================================================
def capture_single_frame(quality=0.8):
    js = Javascript('''
      async function captureFrame(quality) {
        const div = document.createElement('div');
        const btn = document.createElement('button');
        btn.textContent = '📸 Capture Test Frame';
        btn.style.cssText = 'padding: 8px 16px; font-weight: bold; background: #007bff; color: white; border: none; border-radius: 4px; cursor: pointer;';
        div.appendChild(btn);

        const video = document.createElement('video');
        video.style.display = 'block';
        video.style.margin = '10px 0';
        const stream = await navigator.mediaDevices.getUserMedia({video: {width: 640, height: 480}});
        video.srcObject = stream;
        await video.play();

        document.body.appendChild(div);
        div.appendChild(video);

        await new Promise((resolve) => btn.onclick = resolve);

        const canvas = document.createElement('canvas');
        canvas.width = video.videoWidth;
        canvas.height = video.videoHeight;
        canvas.getContext('2d').drawImage(video, 0, 0);
        stream.getVideoTracks()[0].stop();
        div.remove();
        return canvas.toDataURL('image/jpeg', quality);
      }
    ''')
    display(js)
    data = eval_js(f'captureFrame({quality})')
    binary = base64.b64decode(data.split(',')[1])
    return cv2.imdecode(np.frombuffer(binary, dtype=np.uint8), cv2.IMREAD_COLOR)

print("Starting single-frame validation...")
test_frame = capture_single_frame()

# Run inference
results = model.predict(source=test_frame, imgsz=480, conf=0.30, classes=ACTIVE_ONTOLOGY_IDS, device=device, verbose=False)
result = results[0]
annotated = result.plot()

# Display visual output
display(PILImage.fromarray(annotated[..., ::-1]))

# Tabulate detections
print("\n" + "=" * 65)
print(f"{'Detected Class':15} | {'Confidence':10} | {'Rel Area':10} | {'Hazard State':15}")
print("=" * 65)
for box in result.boxes:
    cls_id = int(box.cls[0])
    conf = float(box.conf[0])
    hazard = compute_spatial_hazard(box, test_frame.shape)
    state = "CRITICAL" if hazard["is_critical"] else ("MODERATE" if hazard["is_moderate"] else "LOW")
    print(f"{model.names[cls_id]:15} | {conf:<10.3f} | {hazard['norm_area']:<10.4f} | {state:15}")

Starting single-frame validation...


<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# Cell 5: Real-Time Stream (FAST) - browser-side video + async detection
# ============================================================
# Why the old version was slow: every frame did 3 blocking eval_js round trips
# (pull frame, push annotated JPEG back, speak) -> ~3-6 FPS no matter how fast YOLO is.
# Now: the video plays natively in the browser (smooth), frames are sent to Python
# asynchronously, and Python returns only a tiny JSON of boxes that JS draws on a canvas.
import json, time, base64
import numpy as np, cv2
from IPython.display import display, Javascript, JSON
from google.colab import output

INFER_SIZE     = 320      # try 256 for more speed, 416 for more accuracy
CONF           = 0.35
SEND_WIDTH     = 320      # width of frames sent to Python
JPEG_QUALITY   = 0.5
ALERT_COOLDOWN = 2.5
TARGET_FPS     = 30       # how often frames are sent
MAX_IN_FLIGHT  = 3        # frames allowed in transit at once (hides network latency)
USE_HALF       = str(device).startswith("cuda")   # FP16 on GPU

if str(device) == "cpu":
    print("⚠️ Running on CPU. In Colab: Runtime > Change runtime type > T4 GPU for a big speedup.")

# Warm-up so the first live frame isn't slow
_ = model.predict(np.zeros((240, 320, 3), np.uint8), imgsz=INFER_SIZE, device=device,
                  half=USE_HALF, verbose=False)

_state = {"last_alert": 0.0}

def detect_frame(b64):
    t0 = time.time()
    frame = cv2.imdecode(np.frombuffer(base64.b64decode(b64), np.uint8), cv2.IMREAD_COLOR)
    h, w = frame.shape[:2]
    result = model.predict(frame, imgsz=INFER_SIZE, conf=CONF, classes=ACTIVE_ONTOLOGY_IDS,
                           device=device, half=USE_HALF, verbose=False)[0]

    dets, top_name, top_area, top_critical = [], None, 0.0, False
    for box in result.boxes:
        label = model.names[int(box.cls[0])]
        geom = compute_spatial_hazard(box, frame.shape)
        x1, y1, x2, y2 = geom["bbox"]
        state = "critical" if geom["is_critical"] else ("moderate" if geom["is_moderate"] else "low")
        dets.append({"label": label, "conf": round(float(box.conf[0]), 2), "state": state,
                     "box": [x1 / w, y1 / h, x2 / w, y2 / h]})
        if geom["is_moderate"] and geom["norm_area"] > top_area:
            top_area, top_name, top_critical = geom["norm_area"], label, geom["is_critical"]

    alert, now = None, time.time()
    if top_name and now - _state["last_alert"] > ALERT_COOLDOWN:
        alert = f"Warning, {top_name} ahead" if top_critical else f"{top_name} detected"
        _state["last_alert"] = now

    return JSON({"dets": dets, "alert": alert, "ms": round((now - t0) * 1000)})

output.register_callback("visionaid.detect", detect_frame)

js = """
(async () => {
  if (window.__visionaid_stop) window.__visionaid_stop();   // stop any previous session
  const SEND_W = %d, Q = %f, INTERVAL = %d, MAX_IN = %d;
  const synth = window.speechSynthesis;
  let voice = null;
  const setupVoice = () => {
    const v = synth.getVoices();
    voice = v.find(x => /Google|Natural|Samantha/.test(x.name) && x.lang.startsWith('en'))
         || v.find(x => x.lang.startsWith('en'));
  };
  setupVoice(); synth.onvoiceschanged = setupVoice;

  const wrap = document.createElement('div');
  wrap.style.cssText = 'display:flex;flex-direction:column;align-items:center;gap:8px;';
  const stage = document.createElement('div');
  stage.style.cssText = 'position:relative;';
  const video = document.createElement('video');
  video.muted = true; video.playsInline = true;
  const overlay = document.createElement('canvas');
  overlay.style.cssText = 'position:absolute;left:0;top:0;width:100%%;height:100%%;pointer-events:none;';
  const hud = document.createElement('div');
  hud.style.cssText = 'font:13px monospace;';
  const stopBtn = document.createElement('button');
  stopBtn.textContent = '⏹ Stop Live Pipeline';
  stopBtn.style.cssText = 'padding:6px 14px;background:#dc3545;color:#fff;border:none;border-radius:4px;cursor:pointer;';
  stage.append(video, overlay); wrap.append(stage, hud, stopBtn);
  document.body.appendChild(wrap);

  const stream = await navigator.mediaDevices.getUserMedia({video: {width: 640, height: 480, frameRate: 30}});
  video.srcObject = stream; await video.play();
  video.style.cssText = 'border:2px solid #007bff;border-radius:6px;max-width:100%%;';
  overlay.width = video.videoWidth; overlay.height = video.videoHeight;

  const send = document.createElement('canvas');
  send.width = SEND_W; send.height = Math.round(SEND_W * video.videoHeight / video.videoWidth);
  const sctx = send.getContext('2d'), octx = overlay.getContext('2d');
  const colors = {critical:'#ff3b30', moderate:'#ff9500', low:'#34c759'};

  let running = true, frames = 0, t0 = performance.now();
  window.__visionaid_stop = () => {
    running = false; synth.cancel();
    stream.getTracks().forEach(t => t.stop()); wrap.remove();
  };
  stopBtn.onclick = window.__visionaid_stop;

  function draw(dets) {
    octx.clearRect(0, 0, overlay.width, overlay.height);
    octx.lineWidth = 3; octx.font = '16px sans-serif';
    for (const d of dets) {
      const [x1, y1, x2, y2] = d.box;
      const X = x1 * overlay.width, Y = y1 * overlay.height;
      const W = (x2 - x1) * overlay.width, H = (y2 - y1) * overlay.height;
      octx.strokeStyle = colors[d.state]; octx.strokeRect(X, Y, W, H);
      const txt = d.label + ' ' + d.conf;
      octx.fillStyle = colors[d.state];
      octx.fillRect(X, Math.max(0, Y - 20), octx.measureText(txt).width + 8, 20);
      octx.fillStyle = '#fff'; octx.fillText(txt, X + 4, Math.max(14, Y - 5));
    }
  }

  let inflight = 0, seq = 0, shown = 0;

  async function sendOne() {
    const id = ++seq;
    sctx.drawImage(video, 0, 0, send.width, send.height);          // capture synchronously
    const b64 = send.toDataURL('image/jpeg', Q).split(',')[1];
    inflight++;
    try {
      const res = await google.colab.kernel.invokeFunction('visionaid.detect', [b64], {});
      const out = res.data['application/json'];
      if (id > shown) {                                             // never draw an older result over a newer one
        shown = id;
        draw(out.dets);
        if (out.alert && !synth.speaking) {
          const u = new SpeechSynthesisUtterance(out.alert);
          if (voice) u.voice = voice; u.rate = 1.05; synth.speak(u);
        }
      }
      frames++;
      const dt = (performance.now() - t0) / 1000;
      if (dt > 1) { hud.textContent = (frames / dt).toFixed(1) + ' det/s | infer ' + out.ms + ' ms'; frames = 0; t0 = performance.now(); }
    } catch (e) { console.error(e); }
    finally { inflight--; }
  }

  while (running) {
    if (inflight < MAX_IN) sendOne();                               // fire without awaiting -> pipelined
    await new Promise(r => setTimeout(r, INTERVAL));
  }
})();
""" % (SEND_WIDTH, JPEG_QUALITY, 1000 // TARGET_FPS, MAX_IN_FLIGHT)

display(Javascript(js))
print("Live perception and warning system active (video is smooth; boxes update as fast as the model runs)...")


## 🏁 Engineering Assessment & Conclusion

1. **Experimental Validation:** The perception pipeline demonstrates low-latency inference on single-frame and real-time browser streams using the nano architecture (`yolo26n`), effectively isolating prioritized hazards from background clutter.
2. **Proximity & Warning Separation:** Spatial proximity was decoupled from confidence scores by computing normalized bounding-box area ratios and vertical ground anchors, preventing false confidence-driven alerts.
3. **Roadmap to Production:**
   - Annotate and incorporate negative obstacles (`Stairs`, `Steps`) and doorways into a custom `data.yaml`.
   - Export the model to **ONNX / TensorRT / TFLite** for on-device deployment on edge hardware (Raspberry Pi 5 / Jetson Orin Nano).
   - Integrate stereo-depth or ultrasonic sensor fusion to transition from 2D relative proximity to calibrated 3D metric distance measurements.